# Firecracker RL — Offline Training

Trains 7 models on collected EffiBench transitions:
- **Baselines**: Always-Max, Always-Min, Random, Rule-Based
- **Supervised**: Behavioral Cloning (MLP)
- **Offline RL (discrete)**: CQL, IQL

Dataset: 278 problems × 14 resource configs = 3892 transitions

In [ ]:
!pip install -q torch scikit-learn matplotlib seaborn pandas numpy

In [ ]:
import json, math, random, os
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
print('Imports OK')

## 1. Load Data

Upload  to Kaggle/Colab then set the path below.

In [ ]:
TRANS_PATH = '/kaggle/input/datasets/madihaahmedchowdhury/thesis-madiha-ahmed-chowdhury/effibench_transitions.jsonl'
# If running on Colab:
# TRANS_PATH = '/content/drive/MyDrive/firecracker-rl/effibench_transitions.jsonl'

raw = []
with open(TRANS_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            raw.append(json.loads(line))

print(f'Loaded {len(raw)} transitions from {len(set(t["task_id"] for t in raw))} problems')

In [ ]:
# ── Action configs (14 discrete resource allocations) ──────────────────────────
ACTION_CONFIGS = [
    {'cpu_millicores':  50, 'memory_limit_mb':  32, 'timeout_ms':  1000},
    {'cpu_millicores':  50, 'memory_limit_mb':  48, 'timeout_ms':  2000},
    {'cpu_millicores':  50, 'memory_limit_mb':  64, 'timeout_ms':  3000},
    {'cpu_millicores':  80, 'memory_limit_mb':  48, 'timeout_ms':  1500},
    {'cpu_millicores':  80, 'memory_limit_mb':  64, 'timeout_ms':  2500},
    {'cpu_millicores':  80, 'memory_limit_mb':  96, 'timeout_ms':  4000},
    {'cpu_millicores': 100, 'memory_limit_mb':  64, 'timeout_ms':  2000},
    {'cpu_millicores': 100, 'memory_limit_mb':  96, 'timeout_ms':  3500},
    {'cpu_millicores': 100, 'memory_limit_mb': 128, 'timeout_ms':  5000},
    {'cpu_millicores': 150, 'memory_limit_mb':  64, 'timeout_ms':  3000},
    {'cpu_millicores': 150, 'memory_limit_mb':  96, 'timeout_ms':  5000},
    {'cpu_millicores': 250, 'memory_limit_mb': 128, 'timeout_ms':  5000},
    {'cpu_millicores': 250, 'memory_limit_mb': 192, 'timeout_ms':  8000},
    {'cpu_millicores': 500, 'memory_limit_mb': 256, 'timeout_ms': 10000},
]
N_ACTIONS = len(ACTION_CONFIGS)
CPU_MAX, MEM_MAX, TO_MAX = 500, 256, 10000

def action_to_idx(action):
    key = (action['cpu_millicores'], action['memory_limit_mb'], action['timeout_ms'])
    for i, c in enumerate(ACTION_CONFIGS):
        if (c['cpu_millicores'], c['memory_limit_mb'], c['timeout_ms']) == key:
            return i
    return -1

# ── All 19 state features ───────────────────────────────────────────────────────
FEATURE_COLS = [
    'prompt_token_count',
    'prompt_complexity_score',
    'task_type',
    'has_loops_hint',
    'has_io_hint',
    'example_count',
    'line_count',
    'cyclomatic_complexity',
    'ast_node_count',
    'has_recursion',
    'has_external_calls',
    'max_loop_depth',
    'estimated_complexity',
    'host_cpu_load_1m',
    'host_mem_available_mb',
    'queue_depth',
    'recent_success_rate',
    'recent_mean_cpu_used',
    'recent_mean_mem_used',
]
N_FEATURES = len(FEATURE_COLS)
print(f'{N_ACTIONS} action configs | {N_FEATURES} state features')

In [ ]:
# ── Reward recomputation (matches actions.py exactly) ─────────────────────────
def recompute_reward(t):
    ex     = t['execution']
    action = t['action']

    r_success = 1.0 if ex['exit_code'] == 0 else -2.0
    r_timeout = -5.0 if ex['timed_out'] else 0.0

    wall_ms  = max(ex.get('wall_time_ms', 0) or 0, 1)
    wall_sec = wall_ms / 1000

    cpu_budget_ms = (action['cpu_millicores'] / 1000) * wall_sec * 1000
    cpu_used_ms   = (ex.get('cpu_user_ms', 0) or 0) + (ex.get('cpu_sys_ms', 0) or 0)
    cpu_waste     = max(0.0, (cpu_budget_ms - cpu_used_ms) / max(cpu_budget_ms, 1))

    mem_alloc_kb  = action['memory_limit_mb'] * 1024
    mem_used_kb   = ex.get('mem_peak_kb', 0) or 0
    mem_waste     = max(0.0, (mem_alloc_kb - mem_used_kb) / max(mem_alloc_kb, 1))

    r_resource    = -0.75 * cpu_waste - 0.75 * mem_waste
    r_latency     = -0.2 * math.log(wall_ms / 500 + 1)
    tests_passed  = ex.get('tests_passed')
    r_correctness = 1.0 if tests_passed is True else (-1.0 if tests_passed is False else 0.0)

    return r_success + r_timeout + r_resource + r_latency + r_correctness

# ── Build DataFrame ────────────────────────────────────────────────────────────
rows = []
for t in raw:
    state = t['state']
    row = {
        'task_id':         t['task_id'],
        'action_idx':      action_to_idx(t['action']),
        'reward':          recompute_reward(t),
        'tests_passed':    t['execution']['tests_passed'],
        'exit_code':       t['execution']['exit_code'],
        'timed_out':       t['execution']['timed_out'],
        'cpu_millicores':  t['action']['cpu_millicores'],
        'memory_limit_mb': t['action']['memory_limit_mb'],
        'timeout_ms':      t['action']['timeout_ms'],
        'wall_time_ms':    t['execution'].get('wall_time_ms', 0) or 0,
        'mem_peak_kb':     t['execution'].get('mem_peak_kb', 0) or 0,
        'stress_level':    t.get('meta', {}).get('stress_level', 'unknown'),
    }
    for f in FEATURE_COLS:
        row[f] = float(state.get(f, 0) or 0)
    rows.append(row)

df = pd.DataFrame(rows)
print(f'DataFrame: {df.shape}')
print(df[FEATURE_COLS].describe().round(2))

## 2. Train / Test Split

Split by **problem** (not by transition) — test set contains unseen problems.
80% train / 20% test.

In [ ]:
all_task_ids = df['task_id'].unique()
np.random.seed(42)
test_ids  = set(np.random.choice(all_task_ids, size=int(len(all_task_ids) * 0.2), replace=False))
train_ids = set(all_task_ids) - test_ids

df_train = df[df['task_id'].isin(train_ids)].reset_index(drop=True)
df_test  = df[df['task_id'].isin(test_ids)].reset_index(drop=True)

print(f'Train: {len(train_ids)} problems | {len(df_train)} transitions')
print(f'Test:  {len(test_ids)}  problems | {len(df_test)}  transitions')

# Normalize state features
scaler = StandardScaler()
X_train_all = scaler.fit_transform(df_train[FEATURE_COLS].values)
X_test_all  = scaler.transform(df_test[FEATURE_COLS].values)

def best_action_per_problem(df_sub):
    best = {}
    for tid, g in df_sub.groupby('task_id'):
        best[tid] = int(g.loc[g['reward'].idxmax(), 'action_idx'])
    return best

train_best = best_action_per_problem(df_train)
test_best  = best_action_per_problem(df_test)

prob_train   = df_train.drop_duplicates('task_id').reset_index(drop=True)
prob_test    = df_test.drop_duplicates('task_id').reset_index(drop=True)
X_prob_train = scaler.transform(prob_train[FEATURE_COLS].values)
X_prob_test  = scaler.transform(prob_test[FEATURE_COLS].values)
y_prob_train = np.array([train_best[t] for t in prob_train['task_id']])
y_prob_test  = np.array([test_best[t]  for t in prob_test['task_id']])
print(f'
BC label distribution (train):')
print(pd.Series(y_prob_train).value_counts().sort_index())

## 3. Evaluation Utility

For each model: pick a config per test problem, look up its actual reward from the dataset.

In [ ]:
def evaluate_policy(policy_fn, df_eval, label):
    rewards, successes, oracle_matches = [], [], []
    for tid, group in df_eval.groupby('task_id'):
        state_raw  = group.iloc[0][FEATURE_COLS].values.astype(float)
        state_norm = scaler.transform(state_raw.reshape(1, -1))[0]
        pred_idx   = int(policy_fn(state_norm))
        row        = group[group['action_idx'] == pred_idx]
        if row.empty:
            continue
        r = float(row.iloc[0]['reward'])
        s = 1 if row.iloc[0]['tests_passed'] is True else 0
        oracle_idx   = int(group.loc[group['reward'].idxmax(), 'action_idx'])
        oracle_match = 1 if pred_idx == oracle_idx else 0
        rewards.append(r)
        successes.append(s)
        oracle_matches.append(oracle_match)
    return {
        'model':        label,
        'avg_reward':   float(np.mean(rewards)),
        'success_rate': float(np.mean(successes)),
        'oracle_acc':   float(np.mean(oracle_matches)),
        'n_problems':   len(rewards),
    }

print('Evaluation utility ready.')

## 4. Baselines

In [ ]:
always_max_fn = lambda s: 13
always_min_fn = lambda s: 0
random_fn     = lambda s: random.randint(0, N_ACTIONS - 1)

def rule_based_fn(s):
    orig = dict(zip(FEATURE_COLS, scaler.inverse_transform(s.reshape(1, -1))[0]))
    cc   = orig['cyclomatic_complexity']
    lc   = orig['line_count']
    ld   = orig['max_loop_depth']
    ec   = orig['estimated_complexity']
    if cc > 15 or ec >= 3 or ld >= 4:
        return 11
    elif cc > 8 or lc > 50:
        return 10
    elif cc > 4 or lc > 20:
        return 7
    else:
        return 5

for name, fn in [('Always-Max', always_max_fn), ('Always-Min', always_min_fn),
                 ('Random', random_fn), ('Rule-Based', rule_based_fn)]:
    r = evaluate_policy(fn, df_test, name)
    print(f"{name:<12} | reward={r['avg_reward']:+.3f} | success={r['success_rate']:.1%} | oracle_acc={r['oracle_acc']:.1%}")

## 5. Behavioral Cloning (BC)

Supervised: learn to predict the best config from state features.
Label = argmax(reward) across 14 transitions per problem.

In [ ]:
bc_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    max_iter=500,
    random_state=42,
    learning_rate_init=0.001,
    early_stopping=True,
    validation_fraction=0.1,
)
bc_model.fit(X_prob_train, y_prob_train)

train_acc = accuracy_score(y_prob_train, bc_model.predict(X_prob_train))
test_acc  = accuracy_score(y_prob_test,  bc_model.predict(X_prob_test))
print(f'BC train accuracy: {train_acc:.3f}')
print(f'BC test  accuracy: {test_acc:.3f}')

bc_policy_fn = lambda s: int(bc_model.predict(s.reshape(1, -1))[0])
r = evaluate_policy(bc_policy_fn, df_test, 'BC')
print(f"
BC | reward={r['avg_reward']:+.3f} | success={r['success_rate']:.1%} | oracle_acc={r['oracle_acc']:.1%}")

## 6. CQL — Conservative Q-Learning (Discrete)

Offline RL: Q-network over 14 discrete actions.
Single-step MDP (done=True always) → target = r (no bootstrapping).
CQL penalty prevents overestimating unseen actions.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, n_actions),
        )
    def forward(self, x):
        return self.net(x)

X_cql = scaler.transform(df_train[FEATURE_COLS].values)
a_cql = df_train['action_idx'].values.astype(int)
r_cql = df_train['reward'].values.astype(np.float32)

Xt = torch.FloatTensor(X_cql)
at = torch.LongTensor(a_cql)
rt = torch.FloatTensor(r_cql)

CQL_ALPHA  = 1.0
N_EPOCHS   = 300
BATCH_SIZE = 256

cql_net = QNetwork(N_FEATURES, N_ACTIONS)
cql_opt = optim.Adam(cql_net.parameters(), lr=1e-3)
loader  = DataLoader(TensorDataset(Xt, at, rt), batch_size=BATCH_SIZE, shuffle=True)

for epoch in range(N_EPOCHS):
    cql_net.train()
    total_loss = 0
    for xb, ab, rb in loader:
        q_all   = cql_net(xb)
        q_taken = q_all.gather(1, ab.unsqueeze(1)).squeeze(1)
        bellman = ((q_taken - rb) ** 2).mean()
        cql_pen = (torch.logsumexp(q_all, dim=1) - q_taken).mean()
        loss    = bellman + CQL_ALPHA * cql_pen
        cql_opt.zero_grad()
        loss.backward()
        cql_opt.step()
        total_loss += loss.item()
    if (epoch + 1) % 100 == 0:
        print(f'CQL epoch {epoch+1}/{N_EPOCHS} | loss={total_loss/len(loader):.4f}')

cql_net.eval()
def cql_policy_fn(s):
    with torch.no_grad():
        q = cql_net(torch.FloatTensor(s).unsqueeze(0))
    return int(q.argmax().item())

r = evaluate_policy(cql_policy_fn, df_test, 'CQL')
print(f"
CQL | reward={r['avg_reward']:+.3f} | success={r['success_rate']:.1%} | oracle_acc={r['oracle_acc']:.1%}")

## 7. IQL — Implicit Q-Learning (Discrete)

Offline RL that avoids querying out-of-distribution actions.
Three networks trained jointly:
- **Q-network**: Q(s,a) = r  (single-step MDP, no next-state bootstrap)
- **Value network**: V(s) via expectile regression — estimates high quantile of Q without max over unseen actions
- **Policy**: advantage-weighted cross-entropy — upweights actions where Q(s,a) > V(s)

In [ ]:
# ── IQL hyperparameters ───────────────────────────────────────────────────────
IQL_TAU       = 0.7    # expectile (>0.5 -> optimistic value estimate)
IQL_BETA      = 3.0    # inverse temperature for advantage weighting
IQL_CLIP      = 10.0   # clip exp(beta * A) to avoid exploding weights
N_EPOCHS_IQL  = 300
BATCH_SIZE_IQL = 256

# ── Networks ──────────────────────────────────────────────────────────────────
class IQLQNet(nn.Module):
    def __init__(self, s_dim, n_act):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),    nn.ReLU(),
            nn.Linear(64, n_act),
        )
    def forward(self, x): return self.net(x)

class IQLValueNet(nn.Module):
    def __init__(self, s_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),    nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, x): return self.net(x).squeeze(1)

class IQLPolicyNet(nn.Module):
    def __init__(self, s_dim, n_act):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),    nn.ReLU(),
            nn.Linear(64, n_act),
        )
    def forward(self, x): return self.net(x)  # logits

def expectile_loss(diff, tau):
    weight = torch.where(diff >= 0,
                         torch.full_like(diff, tau),
                         torch.full_like(diff, 1.0 - tau))
    return (weight * diff ** 2).mean()

# ── Data ──────────────────────────────────────────────────────────────────────
X_iql = torch.FloatTensor(scaler.transform(df_train[FEATURE_COLS].values))
A_iql = torch.LongTensor(df_train['action_idx'].values.astype(int))
R_iql = torch.FloatTensor(df_train['reward'].values.astype('float32'))

iql_q      = IQLQNet(N_FEATURES, N_ACTIONS)
iql_v      = IQLValueNet(N_FEATURES)
iql_policy = IQLPolicyNet(N_FEATURES, N_ACTIONS)

opt_q = optim.Adam(iql_q.parameters(),      lr=1e-3)
opt_v = optim.Adam(iql_v.parameters(),      lr=1e-3)
opt_p = optim.Adam(iql_policy.parameters(), lr=1e-3)

loader_iql = DataLoader(TensorDataset(X_iql, A_iql, R_iql),
                        batch_size=BATCH_SIZE_IQL, shuffle=True)

# ── Training ──────────────────────────────────────────────────────────────────
for epoch in range(N_EPOCHS_IQL):
    iql_q.train(); iql_v.train(); iql_policy.train()
    for xb, ab, rb in loader_iql:

        # 1. Q-loss: Q(s, a_taken) -> r  (single-step MDP, done=True always)
        q_all   = iql_q(xb)
        q_taken = q_all.gather(1, ab.unsqueeze(1)).squeeze(1)
        q_loss  = ((q_taken - rb) ** 2).mean()
        opt_q.zero_grad(); q_loss.backward(); opt_q.step()

        # 2. V-loss: expectile regression of (Q(s,a) - V(s))
        with torch.no_grad():
            q_target = iql_q(xb).gather(1, ab.unsqueeze(1)).squeeze(1)
        v_pred = iql_v(xb)
        v_loss = expectile_loss(q_target - v_pred, IQL_TAU)
        opt_v.zero_grad(); v_loss.backward(); opt_v.step()

        # 3. Policy: advantage-weighted cross-entropy
        with torch.no_grad():
            q_sa   = iql_q(xb).gather(1, ab.unsqueeze(1)).squeeze(1)
            v_s    = iql_v(xb)
            adv    = q_sa - v_s
            weight = torch.exp(torch.clamp(IQL_BETA * adv, -IQL_CLIP, IQL_CLIP))
        logits = iql_policy(xb)
        log_p  = torch.nn.functional.log_softmax(logits, dim=1)
        log_pa = log_p.gather(1, ab.unsqueeze(1)).squeeze(1)
        p_loss = -(weight * log_pa).mean()
        opt_p.zero_grad(); p_loss.backward(); opt_p.step()

    if (epoch + 1) % 100 == 0:
        print(f'IQL epoch {epoch+1}/{N_EPOCHS_IQL} | '
              f'q={q_loss.item():.4f} v={v_loss.item():.4f} p={p_loss.item():.4f}')

iql_policy.eval()
def iql_policy_fn(s):
    with torch.no_grad():
        logits = iql_policy(torch.FloatTensor(s).unsqueeze(0))
    return int(logits.argmax().item())

r = evaluate_policy(iql_policy_fn, df_test, 'IQL')
print(f"\nIQL | reward={r['avg_reward']:+.3f} | success={r['success_rate']:.1%} | oracle_acc={r['oracle_acc']:.1%}")


## 7. Full Comparison

In [ ]:
all_models = [
    ('Always-Max',  always_max_fn),
    ('Always-Min',  always_min_fn),
    ('Random',      random_fn),
    ('Rule-Based',  rule_based_fn),
    ('BC',          bc_policy_fn),
    ('CQL',         cql_policy_fn),
    ('IQL',         iql_policy_fn),
]

results = [evaluate_policy(fn, df_test, name) for name, fn in all_models]
results_df = pd.DataFrame(results).set_index('model')
results_df = results_df[['avg_reward', 'success_rate', 'oracle_acc', 'n_problems']]
results_df.columns = ['Avg Reward', 'Success Rate', 'Oracle Acc', 'N Problems']
print(results_df.round(3).to_string())

## 8. Plots

In [ ]:
models     = results_df.index.tolist()
colors_bar = ['#e74c3c' if v < 0 else '#2ecc71' for v in results_df['Avg Reward']]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison — Test Set', fontsize=14, fontweight='bold')

axes[0].barh(models, results_df['Avg Reward'], color=colors_bar, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Average Reward')
axes[0].set_xlabel('Reward')

axes[1].barh(models, results_df['Success Rate'], color='#3498db', edgecolor='white')
axes[1].set_title('Success Rate')
axes[1].set_xlim(0, 1)
axes[1].set_xlabel('Fraction')

axes[2].barh(models, results_df['Oracle Acc'], color='#9b59b6', edgecolor='white')
axes[2].set_title('Oracle Accuracy
(exact config match)')
axes[2].set_xlim(0, 1)
axes[2].set_xlabel('Fraction')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, (name, fn) in enumerate(all_models):
    counts = defaultdict(int)
    for tid, group in df_test.groupby('task_id'):
        s = scaler.transform(group.iloc[0][FEATURE_COLS].values.reshape(1, -1))[0]
        counts[fn(s)] += 1
    vals = [counts.get(j, 0) for j in range(N_ACTIONS)]
    axes[i].bar(range(N_ACTIONS), vals, color='#3498db', edgecolor='white')
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xticks(range(N_ACTIONS))
    axes[i].set_xticklabels([str(j) for j in range(N_ACTIONS)], fontsize=7)
    axes[i].set_xlabel('Config index')
    axes[i].set_ylabel('Count')

axes[-1].set_visible(False)
plt.suptitle('Config Choices per Model (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('config_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
df_grouped = df.groupby('action_idx')['reward'].agg(['mean', 'std']).reset_index()
ax.bar(df_grouped['action_idx'], df_grouped['mean'],
       yerr=df_grouped['std'], capsize=4, color='#2ecc71', edgecolor='white', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(range(N_ACTIONS))
ax.set_xticklabels([f"cfg{i}
{c['cpu_millicores']}mc/{c['memory_limit_mb']}MB"
                    for i, c in enumerate(ACTION_CONFIGS)], fontsize=7, rotation=30)
ax.set_title('Mean Reward ± Std by Config (All Problems)', fontweight='bold')
ax.set_ylabel('Reward')
plt.tight_layout()
plt.savefig('reward_by_config.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
results_df.to_csv('model_results.csv')
print('Saved model_results.csv')
print('
Final results:')
print(results_df.sort_values('Avg Reward', ascending=False).round(3).to_string())